In [0]:
display(spark.table("recruitment.silver_claims").limit(10))

In [0]:
from pyspark.sql.functions import (
    col,
    count,
    sum,
    avg,
    when,
    round
)

silver_table = "recruitment.silver_claims"

silver_df = (
    spark.read
    .format("delta")
    .table(silver_table)
)

In [0]:
gold_daily_df = (
    silver_df
    .groupBy("incident_date")
    .agg(
        count("*").alias("total_claims"),

        round(
            sum("total_claim_amount"), 2
        ).alias("total_claim_amount"),

        round(
            avg("total_claim_amount"), 2
        ).alias("avg_claim_amount"),

        sum(
            when(col("fraud_reported") == "Y", 1).otherwise(0)
        ).alias("fraudulent_claims"),

        round(
            avg("age"), 2
        ).alias("avg_customer_age")
    )
    .withColumn(
        "fraud_rate",
        round(
            col("fraudulent_claims") /
            col("total_claims") * 100,
            2
        )
    )
)

In [0]:
from pyspark.sql.functions import current_timestamp

gold_daily_df = gold_daily_df.withColumn(
    "updated_at",
    current_timestamp()
)


In [0]:
gold_daily_table = "recruitment.gold_claims_daily"

(
    gold_daily_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(gold_daily_table)
)

In [0]:
display(
    spark.table(gold_daily_table)
    .orderBy("incident_date")
)

In [0]:
%sql
ALTER TABLE recruitment.gold_claims_daily
CLUSTER BY (incident_date);

OPTIMIZE recruitment.gold_claims_daily

##Gold 2

In [0]:
gold_state_df = (
    silver_df
    .groupBy("policy_state")
    .agg(
        count("*").alias("total_claims"),
        round(sum("total_claim_amount"), 2).alias("total_claim_amount"),
        round(avg("total_claim_amount"), 2).alias("avg_claim_amount"),
        sum(
            when(col("fraud_reported") == "Y", 1).otherwise(0)
        ).alias("fraudulent_claims")
    )
    .withColumn(
        "fraud_rate",
        round(
            col("fraudulent_claims") /
            col("total_claims") * 100,
            2
        )
    )
)

In [0]:
gold_state_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("recruitment.gold_claims_by_state")

In [0]:
display(
    spark.table("recruitment.gold_claims_by_state")
)

##Gold 3

In [0]:
gold_fraud_df = (
    silver_df
    .groupBy(
        "incident_type",
        "incident_severity"
    )
    .agg(
        count("*").alias("total_claims"),
        sum(
            when(col("fraud_reported") == "Y", 1).otherwise(0)
        ).alias("fraudulent_claims"),
        round(avg("total_claim_amount"), 2)
            .alias("avg_claim_amount"),
        round(sum("total_claim_amount"), 2)
            .alias("total_claim_amount")
    )
    .withColumn(
        "fraud_rate",
        round(
            col("fraudulent_claims") /
            col("total_claims") * 100,
            2
        )
    )
)

In [0]:
gold_fraud_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("recruitment.gold_fraud_analysis")

In [0]:
%sql
ALTER TABLE recruitment.gold_fraud_analysis
CLUSTER BY (incident_type, incident_severity);

In [0]:
display(
    spark.table("recruitment.gold_fraud_analysis")
    .orderBy("incident_type","incident_severity")
)